# Assignment 1

Computer Vision — COMP9320

In [ ]:
# Import the libraries used throughout the assignment.
%matplotlib inline
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the input and output paths relative to the project or notebook directory.
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "data" / "uni.jpeg").is_file():
    ASSIGNMENT_DIR = ASSIGNMENT_DIR / "Assignments" / "Assign_1"
DATA_DIR = ASSIGNMENT_DIR / "data"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"

## Task 1: Image Loading and Grayscale Conversion

### 1. Use OpenCV to read the image uni.jpeg.

In [ ]:
# Read the original color image in OpenCV BGR format.
image_path = DATA_DIR / "uni.jpeg"
original_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
if original_bgr is None:
    raise FileNotFoundError(f"Unable to read image: {image_path}")

### 2. Display the original image and report its properties.

#### Spatial dimensions

- **Pixel:** one spatial sample of a digital image; a color pixel contains values for its color channels.
- **Spatial dimensions:** image height (rows) and width (columns), measured in pixels. For a color array, `shape` is `(height, width, channels)`; `shape[:2]` selects only the spatial dimensions.
- **BGR and RGB:** channel orders for blue, green, and red values. OpenCV loads this image in BGR order; Matplotlib expects RGB. Reordering channels preserves the spatial dimensions and displays the intended colors.

In [ ]:
# Display the original image in RGB order and report its height and width.
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
height, width = original_bgr.shape[:2]
print(f"Spatial dimensions (height x width): {height} x {width} pixels")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(original_rgb)
ax.set_title("Original image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
plt.tight_layout()
plt.show()

**Observation:** The image has 476 rows and 645 columns, so it is wider than it is tall. Array indexing uses `(row, column)`; the plot uses columns horizontally and rows vertically.

**References — documentation and further reading:**
- [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html) — pixel values, BGR order, and array shape.
- [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — RGB display and image coordinates.

#### Number of color channels

A **color channel** is a two-dimensional array containing one color component at every pixel. This image has blue, green, and red channels. OpenCV stores them in BGR order; each pixel therefore contains three values. The third element of `shape` gives the channel count, whereas `ndim` counts array axes.

In [ ]:
# Report the color channel count, channel order, and array structure.
channel_count = original_bgr.shape[2]
print(f"Number of color channels: {channel_count}")
print("OpenCV channel order: BGR (Blue, Green, Red)")
print("Display channel order: RGB (Red, Green, Blue)")
print(f"Image array shape (height, width, channels): {original_bgr.shape}")
print(f"Number of array axes: {original_bgr.ndim}")
for index, name in enumerate(("Blue", "Green", "Red")):
    print(f"Channel {index}: {name}; spatial shape: {original_bgr[:, :, index].shape}")

**Observation:** There are three color channels, each with 476 rows and 645 columns. The channel axis stores color components; it is not an additional spatial dimension.

**Reference — documentation:** [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html), sections on image properties and splitting channels.

#### Data type and numerical intensity range

- **Data type (`dtype`):** the representation used to store each channel value. `uint8` is an unsigned 8-bit integer: it represents 256 integer levels from 0 to 255.
- **Channel intensity:** the stored value of one color component at a pixel. Zero is the minimum component level; 255 is the maximum. A single channel value is not the overall brightness of a color pixel.
- **Representable range:** the limits allowed by the data type, obtained with `np.iinfo`. **Observed range:** the minimum and maximum actually present in the image, obtained with `min()` and `max()`. These ranges need not be equal.

In [ ]:
# Report the data type, representable limits, and observed channel intensities.
dtype_limits = np.iinfo(original_bgr.dtype)
print(f"Data type: {original_bgr.dtype}")
print(f"Bits per channel value: {dtype_limits.bits}")
print(f"Representable intensity range: [{dtype_limits.min}, {dtype_limits.max}]")
print(f"Representable intensity levels: {int(dtype_limits.max) - int(dtype_limits.min) + 1}")
print(f"Observed intensity range across all channels: [{original_bgr.min()}, {original_bgr.max()}]")
for index, name in enumerate(("Blue", "Green", "Red")):
    channel = original_bgr[:, :, index]
    print(f"Observed {name} channel range: [{channel.min()}, {channel.max()}]")

**Observation:** The image uses `uint8`. The observed range is [0, 255] both overall and in each color channel. Each channel contains at least one minimum and one maximum value; this does not establish that all 256 levels occur or that a completely black or white pixel exists.

**Reference — documentation:** [NumPy: numpy.iinfo](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) — integer limits and bit depth.

#### Memory requirements

- **Byte:** 8 bits. A `uint8` channel value occupies one byte (`itemsize`).
- **Array data memory:** the bytes occupied by the decoded pixel values (`nbytes`), excluding Python object metadata and other arrays. For this image: $M = H \times W \times C \times \text{bytes per channel value}$.
- **Units:** 1 KiB = 1,024 bytes; 1 MiB = 1,048,576 bytes; 1 MB = 1,000,000 bytes.
- **JPEG file size:** compressed storage on disk; it differs from the decoded array memory.

In [ ]:
# Calculate decoded image memory and compare it with the compressed JPEG file size.
pixel_count = height * width
bytes_per_value = original_bgr.itemsize
expected_bytes = pixel_count * channel_count * bytes_per_value
image_bytes = original_bgr.nbytes
jpeg_bytes = image_path.stat().st_size
print(f"Number of spatial pixels: {pixel_count:,}")
print(f"Number of stored channel values: {original_bgr.size:,}")
print(f"Bytes per channel value: {bytes_per_value}")
print(f"Bytes per color pixel: {channel_count * bytes_per_value}")
print(f"Memory calculation: {height} x {width} x {channel_count} x {bytes_per_value} = {expected_bytes:,} bytes")
print(f"Decoded array data (nbytes): {image_bytes:,} bytes")
print(f"Decoded array data: {image_bytes / 1024:.2f} KiB; {image_bytes / 1024**2:.4f} MiB; {image_bytes / 10**6:.5f} MB")
print(f"Compressed JPEG file: {jpeg_bytes:,} bytes ({jpeg_bytes / 1024:.2f} KiB)")
print(f"Calculated memory matches nbytes: {expected_bytes == image_bytes}")

**Observation:** The decoded BGR pixel data occupies 921,060 bytes, whereas the JPEG file occupies 67,589 bytes. JPEG stores a compressed representation. The reported array memory excludes the separate RGB display copy, plotting buffers, and Python object overhead.

**References — documentation:** [NumPy: ndarray.nbytes](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html) and [ndarray.itemsize](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.itemsize.html).